# ANÁLISIS DE ESTÁTICA CON PYTHON

## TEMA: Descomposición de Vectores de Fuerza en 2D

> **Autor:** [Tu Nombre Completo]
> **Fecha:** 9 de octubre de 2025
> **Asignatura:** Física Estática
> **Fuente Principal:** Engineering Mechanics: Statics, 2nd Ed. - William F. Riley & Leroy D. Sturges
> **Capítulo de Referencia:** Capítulo 2 - Vectors

---

### **1. Objetivo del Notebook**

El propósito de este documento es analizar y resolver problemas relacionados con la descomposición de vectores de fuerza en un plano bidimensional. Se implementarán funciones en Python para calcular las componentes rectangulares (Fx, Fy) a partir de la magnitud y dirección de un vector, y se utilizará Matplotlib para visualizar los resultados gráficamente, verificando los conceptos teóricos presentados por Riley & Sturges.

### **2. Conceptos Fundamentales**

* **Vector de Fuerza:** Representación de una fuerza con magnitud, dirección y sentido.
* **Componentes Rectangulares:** Proyecciones de un vector sobre los ejes cartesianos (X, Y).
* **Trigonometría del Triángulo Rectángulo:** Aplicación de seno, coseno y tangente para la descomposición.
* **Notación Vectorial Cartesiana:** Expresión de un vector en términos de sus componentes y los vectores unitarios $\hat{\imath}$ y $\hat{\jmath}$.

### **3. Ecuaciones Clave**

Las componentes de un vector de fuerza $\vec{F}$ con magnitud $F$ y un ángulo $\theta$ medido desde el eje X positivo son:

$$ F_x = F \cos(\theta) $$
$$ F_y = F \sin(\theta) $$

Y el vector se expresa como:

$$ \vec{F} = F_x \hat{\imath} + F_y \hat{\jmath} $$

---

### **4. Configuración del Entorno de Trabajo**

In [1]:
import numpy as np  # NumPy para arrays y cálculos numéricos
import matplotlib.pyplot as plt  # Interfaz de Matplotlib para graficar
from mpl_toolkits.mplot3d import Axes3D  # Soporte para ejes 3D (proyección '3d')
from matplotlib.lines import Line2D  # Util para crear proxies de leyenda

# Estilo de la figura: hace el fondo y la paleta más agradables
plt.style.use('seaborn-v0_8-darkgrid')  # Usa un estilo tipo seaborn si está disponible
import matplotlib  # Importamos matplotlib base para controlar el backend gráfico
# Intentar forzar un backend interactivo para abrir la figura en ventana separada (Windows)
# Se prueban backends comunes: Qt5Agg, QtAgg, TkAgg, WXAgg (en ese orden)
preferred_backends = ['Qt5Agg', 'QtAgg', 'TkAgg', 'WXAgg']  # Lista de opciones de backend
for b in preferred_backends:  # Itera y prueba cada backend disponible
    try:
        matplotlib.use(b, force=True)  # Fuerza usar el backend b
        print('matplotlib backend:', matplotlib.get_backend())  # Muestra qué backend quedó activo
        break  # Si tuvo éxito, deja de probar más backends
    except Exception:
        continue  # Si falla (p. ej. falta PyQt5), prueba el siguiente

# --- 1. Definición de los datos del problema ---
A = np.array([1.2, -1.5, 2.4])  # Coordenadas del punto A (x, y, z) en metros
B = np.array([-1.8, -2.1, 2.1])  # Coordenadas del punto B
C = np.array([-1.2, 1.8, 0.9])  # Coordenadas del punto C
peso_magnitud = 500.0  # Magnitud del peso en newtons
W = np.array([0.0, 0.0, -peso_magnitud])  # Vector peso apuntando en -z

# --- 2. Direcciones unitarias desde D (origen) ---
r_DA, r_DB, r_DC = A, B, C  # Vectores de posición desde el origen D hacia A, B, C
mag_DA, mag_DB, mag_DC = np.linalg.norm(r_DA), np.linalg.norm(r_DB), np.linalg.norm(r_DC)  # Magnitudes
u_DA, u_DB, u_DC = r_DA / mag_DA, r_DB / mag_DB, r_DC / mag_DC  # Vectores unitarios (dirección)

# --- 3. Resolver el sistema de equilibrio ---
A_matrix = np.array([u_DA, u_DB, u_DC]).T  # Matriz de coeficientes (3x3)
b_vector = -W  # Lado derecho: -W (porque T_A*u_A + T_B*u_B + T_C*u_C = -W)
try:
    tensiones = np.linalg.solve(A_matrix, b_vector)  # Resuelve Ax = b para tensiones
    T_A, T_B, T_C = tensiones  # Extrae tensiones individuales
except np.linalg.LinAlgError:
    raise RuntimeError('La matriz de direcciones es singular: revise la geometría de los cables')  # Mensaje claro si no hay solución

print('--- Resultados de las Tensiones ---')  # Encabezado de salida
print(f'La tensión en el cable A (T_A) es: {T_A:.2f} N')  # Imprime T_A con 2 decimales
print(f'La tensión en el cable B (T_B) es: {T_B:.2f} N')  # Imprime T_B
print(f'La tensión en el cable C (T_C) es: {T_C:.2f} N')  # Imprime T_C

# --- 4. Preparar vectores de fuerza para graficar ---
T_A_vec, T_B_vec, T_C_vec = T_A * u_DA, T_B * u_DB, T_C * u_DC  # Vectores de fuerza reales (N)
vectors = [T_A_vec, T_B_vec, T_C_vec, W]  # Lista de fuerzas a graficar
labels = ['T_A', 'T_B', 'T_C', 'W (peso)']  # Etiquetas para anotaciones/leyenda
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']  # Colores para cada vector

# Escalado visual: hace las fuerzas visibles en la misma escena que las coordenadas (m)
coords = np.vstack([A, B, C, np.array([0.0, 0.0, 0.0])])  # Recopila coordenadas para calcular límites
coord_span = coords.max(axis=0) - coords.min(axis=0)  # Rango por eje (x,y,z)
max_coord_span = max(coord_span.max(), 1e-6)  # Evita división por cero
max_force = max(np.linalg.norm(v) for v in vectors)  # Máxima magnitud de fuerza
scale = 0.45 * max_coord_span / (max_force if max_force != 0 else 1.0)  # Factor N→m sólo para visualización

# --- 5. Utilidad: ejes 3D con escala igual ---
def set_axes_equal(ax):  # Función para forzar proporciones iguales en ejes 3D
    x_limits = ax.get_xlim3d()  # Límite actual X
    y_limits = ax.get_ylim3d()  # Límite actual Y
    z_limits = ax.get_zlim3d()  # Límite actual Z
    x_range = abs(x_limits[1] - x_limits[0])  # Rango X
    x_middle = np.mean(x_limits)  # Centro X
    y_range = abs(y_limits[1] - y_limits[0])  # Rango Y
    y_middle = np.mean(y_limits)  # Centro Y
    z_range = abs(z_limits[1] - z_limits[0])  # Rango Z
    z_middle = np.mean(z_limits)  # Centro Z
    plot_radius = 0.5 * max([x_range, y_range, z_range])  # Radio basado en el mayor rango
    ax.set_xlim3d([x_middle - plot_radius, x_middle + plot_radius])  # Fija límite X
    ax.set_ylim3d([y_middle - plot_radius, y_middle + plot_radius])  # Fija límite Y
    ax.set_zlim3d([z_middle - plot_radius, z_middle + plot_radius])  # Fija límite Z

# --- 6. Graficar con estilo elegante ---
fig = plt.figure(figsize=(10, 8))  # Tamaño de la figura en pulgadas
ax = fig.add_subplot(111, projection='3d')  # Crea ejes 3D
origin = np.array([0.0, 0.0, 0.0])  # Coordenadas del origen D

# Puntos ancla y origen
ax.scatter(*A, color=colors[0], s=80, depthshade=True)  # Punto A
ax.text(*(A + 0.05), 'A', color=colors[0], fontsize=11, weight='bold')  # Etiqueta A
ax.scatter(*B, color=colors[1], s=80, depthshade=True)  # Punto B
ax.text(*(B + 0.05), 'B', color=colors[1], fontsize=11, weight='bold')  # Etiqueta B
ax.scatter(*C, color=colors[2], s=80, depthshade=True)  # Punto C
ax.text(*(C + 0.05), 'C', color=colors[2], fontsize=11, weight='bold')  # Etiqueta C
ax.scatter(*origin, color='k', s=40)  # Punto origen D
ax.text(*(origin + np.array([0.02, 0.02, 0.02])), 'D (origen)', color='k')  # Etiqueta D

# Dibujar cada vector: línea + cabeza con quiver (escalado para visualización)
for vec, label, color in zip(vectors, labels, colors):  # Itera fuerzas/etiquetas/colores
    end = vec * scale  # Extremo visual escalado
    # Línea desde el origen hasta el extremo (mejor visibilidad que solo quiver)
    ax.plot([0, end[0]], [0, end[1]], [0, end[2]], color=color, linewidth=2.4, alpha=0.9)  # Línea
    # Flecha (cabeza)
    ax.quiver(0, 0, 0, end[0], end[1], end[2], color=color, linewidth=1.6, arrow_length_ratio=0.08, normalize=False)  # Cabeza
    # Anotación con la magnitud real de la fuerza
    mag = np.linalg.norm(vec)  # Magnitud en N (sin escalar)
    ax.text(end[0] * 1.05, end[1] * 1.05, end[2] * 1.05, f'{label}: {mag:.1f} N', color=color, fontsize=10)  # Texto

# Leyenda manual con proxies para que muestre magnitud y color
proxies = [Line2D([0], [0], color=c, lw=3) for c in colors]  # Objetos ficticios para la leyenda
ax.legend(proxies, labels, loc='upper right')  # Añade la leyenda

# Etiquetas y título
ax.set_xlabel('X (m)')  # Etiqueta eje X
ax.set_ylabel('Y (m)')  # Etiqueta eje Y
ax.set_zlabel('Z (m)')  # Etiqueta eje Z
ax.set_title('Diagrama 3D: Tensiones en cables y peso (escalado para visualización)')  # Título

# Ajustes visuales finales: mismos límites y escala uniforme
ax.auto_scale_xyz(coords[:,0], coords[:,1], coords[:,2])  # Escala automática inicial
set_axes_equal(ax)  # Fuerza proporciones iguales en los 3 ejes
plt.tight_layout()  # Ajusta márgenes para evitar solapamientos
# Mostrar en ventana independiente (bloqueante para mantener la ventana abierta hasta cerrarla)
plt.show(block=True)  # Muestra la figura en una ventana separada y bloquea hasta cerrarla


matplotlib backend: Qt5Agg
--- Resultados de las Tensiones ---
La tensión en el cable A (T_A) es: 459.25 N
La tensión en el cable B (T_B) es: 32.43 N
La tensión en el cable C (T_C) es: 317.22 N
